# GeoShapley_Plant Level

## I. Prepare the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, make_scorer, cohen_kappa_score, RocCurveDisplay

from sklearn.preprocessing import MinMaxScaler, RobustScaler

from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, ElasticNet
from sklearn.ensemble import StackingClassifier, GradientBoostingRegressor, RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier, MLPRegressor

import lightgbm as lgb
import xgboost as xgb

from sklearn.inspection import PartialDependenceDisplay, partial_dependence, permutation_importance

from sklearn.datasets import load_iris, make_moons

In [ ]:
re_data = pd.read_csv(r'D:\Data\re_data.csv')
production_mode = pd.read_csv(r'D:\Data\plantmode_ml.csv')
re_data = pd.merge(re_data, production_mode, on='name_prod', how='left')
re_data = re_data.drop(['name_book', 'products'], axis=1)
re_data = re_data.dropna()

In [ ]:
name_prod_city = pd.read_excel(r'D:\Data\name_prod_city.xlsx')

In [ ]:
merged_df = pd.merge(re_data, name_prod_city, on='name_prod', how='inner')

In [ ]:
merged_df.to_csv(r'D:\Data\plants_list.csv', index=False)

In [ ]:
name_prod_df = merged_df[['name_prod']].drop_duplicates()

name_prod_df.to_csv(r'D:\Data\name_prod_only.csv', index=False)

In [ ]:
X_plant = re_data.drop(['Iron_Prod', 'Steel_Prod', 'name_prod', 'IDCode', 'plant_id'], axis=1)
y_needed_plant = np.log1p(re_data['Steel_Prod']) 

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.neural_network import MLPRegressor
import numpy as np
import pandas as pd

def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def calculate_wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

X_train, X_test, y_train, y_test = train_test_split(X_plant, y_needed_plant, test_size=0.2, random_state=1)

pollutants = ['Plant_CO_MEAN', 'Plant_NO2_MEAN', 'Plant_PM2_5_MEAN', 'Plant_PM10_MEAN', 
              'Plant_SO2_MEAN', 'Plant_LSTA_MEAN', 'Plant_LSTT_MEAN', 'Plant_NTL_MEAN', 'Plant_O3_MEAN']

other_features = [col for col in X_plant.columns if col not in pollutants + ['Longitude', 'Latitude']]

preprocessor = ColumnTransformer(
    transformers=[
        ('pollutants', StandardScaler(), pollutants),  
        ('spatial', RobustScaler(), ['Longitude', 'Latitude']),  
        ('others', StandardScaler(), other_features)  
    ])

base_models = [
    ('lasso', Lasso(alpha=0.01, random_state=1)),
    ('lightgbm', lgb.LGBMRegressor(objective='regression', num_leaves=5, learning_rate=0.05, n_estimators=720)),
    ('random_forest', RandomForestRegressor(max_depth=4, max_features=9, n_estimators=300, random_state=5)),
    ('xgboost', xgb.XGBRegressor(colsample_bytree=0.6, learning_rate=0.1, max_depth=6, n_estimators=300, random_state=5))
]

meta_model = LinearRegression()
stacking_model = StackingRegressor(estimators=base_models, final_estimator=meta_model, cv=5)


models = {
    "Stacking Model": make_pipeline(preprocessor, stacking_model),
    "Neural Network": make_pipeline(preprocessor, MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam', max_iter=500, random_state=1))
}


results_plant = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results_plant.append({
        "Model": name,
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": calculate_rmse(y_test, y_pred),
        "MAPE": calculate_mape(y_test, y_pred),
        "WMAPE": calculate_wmape(y_test, y_pred),
        "R2 Score": r2_score(y_test, y_pred)
    })


results_df_plant = pd.DataFrame(results_plant)

results_df_plant

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_plant, y_needed_plant, test_size=0.2, random_state=1)

pollutants = ['Plant_CO_MEAN', 'Plant_NO2_MEAN', 'Plant_PM2_5_MEAN', 'Plant_PM10_MEAN', 
              'Plant_SO2_MEAN', 'Plant_LSTA_MEAN', 'Plant_LSTT_MEAN', 'Plant_NTL_MEAN', 'Plant_O3_MEAN']

other_features = [col for col in X_plant.columns if col not in pollutants + ['Longitude', 'Latitude']]

preprocessor = ColumnTransformer(
    transformers=[
        ('pollutants', StandardScaler(), pollutants), 
        ('spatial', RobustScaler(), ['Longitude', 'Latitude']),  
        ('others', StandardScaler(), other_features)  
    ])

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

xgboost_model = xgb.XGBRegressor(
    colsample_bytree=0.6, 
    learning_rate=0.1, 
    max_depth=6, 
    n_estimators=300, 
    random_state=5
)

kf = KFold(n_splits=5, shuffle=True, random_state=1)
cv_scores = cross_val_score(xgboost_model, X_train_preprocessed, y_train, cv=kf, scoring='r2')

print("5-Fold Cross-Validation R2 Scores:", cv_scores)
print("Mean CV R2 Score:", np.mean(cv_scores))

xgboost_model.fit(X_train_preprocessed, y_train)

y_pred = xgboost_model.predict(X_test_preprocessed)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Test MSE: {mse}")
print(f"Test RMSE: {rmse}")
print(f"Test R2 Score: {r2}")

In [ ]:
xgboost_model.fit(X_train, y_train)
xgboost_model.score(X_test, y_test)
y_pred = xgboost_model.predict(X_test)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.kdeplot(y_test, color='blue', fill=True, label='Actual')
sns.kdeplot(y_pred, color='green', fill=True, label='Predicted')
plt.title('Density of Actual vs Predicted Values')
plt.xlabel('Values (10,000 tones)')
plt.legend()

original_ticks = [1, 10, 100]  
log_ticks = [np.log(tick) for tick in original_ticks]  
plt.xticks(log_ticks, original_ticks) 

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred, alpha=0.5, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Actual vs Predicted Values')
plt.xlabel('Actual Values (10,000 tones)')
plt.ylabel('Predicted Values (10,000 tones)')

plt.xticks(log_ticks, original_ticks)  
plt.yticks(log_ticks, original_ticks)  

plt.tight_layout()
plt.savefig(r'D:\Figures\predicted_plant_steel_distribution_adj_realnumber.pdf', format='pdf', dpi=800)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
import xgboost as xgb


def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


xgboost_model = make_pipeline(
    preprocessor, 
    xgb.XGBRegressor(
        colsample_bytree=0.6,  
        learning_rate=0.1,     
        max_depth=6,           
        n_estimators=300,      
        subsample=0.8,         
        alpha=0.2,            
        reg_lambda=0.5,        
        random_state=5        
    )
)


kf = KFold(n_splits=5, shuffle=True, random_state=1)
xgboost_rmse_scores = []


final_model = None
X_test_fold_final = None
y_test_fold_final = None

for train_idx, test_idx in kf.split(X_plant):
    X_train_fold = X_plant.iloc[train_idx]
    y_train_fold = y_needed_plant.iloc[train_idx]
    X_test_fold = X_plant.iloc[test_idx]
    y_test_fold = y_needed_plant.iloc[test_idx]
    
    xgboost_model.fit(X_train_fold, y_train_fold)
    y_pred_fold = xgboost_model.predict(X_test_fold)
    rmse_fold = calculate_rmse(y_test_fold, y_pred_fold)
    xgboost_rmse_scores.append(rmse_fold)
    

    final_model = xgboost_model
    X_test_fold_final = X_test_fold
    y_test_fold_final = y_test_fold


print(f"Average RMSE across folds: {np.mean(xgboost_rmse_scores):.4f}")
xgboost_regressor = final_model.named_steps['xgbregressor']

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.Explainer(xgboost_regressor, preprocessor.transform(X_train_fold))


shap_values = explainer(preprocessor.transform(X_test_fold_final))


plt.figure()  
shap.summary_plot(
    shap_values, 
    preprocessor.transform(X_test_fold_final), 
    feature_names=X_plant.columns,
    show=False  
)

plt.tight_layout()
output_path = r'D:\Figures\PlantLevel\SHAP\SHAP.pdf'
plt.savefig(output_path, format='pdf', dpi=800)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

shap.summary_plot(
    shap_values, 
    X_test, 
    feature_names=X_plant.columns, 
    plot_type="bar",
    show=False  
)

plt.xlabel("SHAP (average impact on the outcome)", fontsize=12)

output_path = r'D:\Figures\shap_feature_importance_plot.pdf'
plt.savefig(output_path, format="pdf", dpi=800, bbox_inches="tight")

## II. Bootstrap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from tqdm import tqdm

def bootstrap_shap(pipeline_model, X_train, y_train, X_test, n_bootstraps=50, save_path=None):
   
    X_train = X_train.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

   
    preprocessor = pipeline_model.named_steps['columntransformer']
    xgb_regressor = pipeline_model.named_steps['xgbregressor']

 
    explainer_shap = shap.TreeExplainer(xgb_regressor)

  
    X_train_transformed = preprocessor.transform(X_train)
    X_test_transformed = preprocessor.transform(X_test)

    shap_bootstrap_list = []

    for i in tqdm(range(n_bootstraps), desc="Bootstrapping SHAP Values"):
        sample_indices = np.random.choice(range(len(X_train)), size=len(X_train), replace=True)
        X_sampled = X_train_transformed[sample_indices]
        y_sampled = y_train.iloc[sample_indices]

        xgb_regressor_clone = xgb_regressor.__class__(**xgb_regressor.get_params())
        xgb_regressor_clone.fit(X_sampled, y_sampled)

        explainer_shap_clone = shap.TreeExplainer(xgb_regressor_clone)
        shap_values = explainer_shap_clone.shap_values(X_test_transformed)
        shap_bootstrap_list.append(shap_values)

        if save_path and (i + 1) % 10 == 0:
            np.save(save_path, np.array(shap_bootstrap_list))

    shap_bootstrap_array = np.array(shap_bootstrap_list)

    if save_path:
        np.save(save_path, shap_bootstrap_array)

    return shap_bootstrap_array

save_path = "shap_bootstrap_intermediate.npy"
shap_bootstrap_values = bootstrap_shap(final_model, X_plant, y_needed_plant, X_test_fold_final, n_bootstraps=50, save_path=save_path)

mean_shap = np.abs(shap_bootstrap_values).mean(axis=1).mean(axis=0)
lower_bound = mean_shap - np.percentile(np.abs(shap_bootstrap_values).mean(axis=1), axis=0, q=2.5)
upper_bound = np.percentile(np.abs(shap_bootstrap_values).mean(axis=1), axis=0, q=97.5) - mean_shap

df_mean_shap = pd.DataFrame(
    {
        "Feature": np.array(X_plant.columns),
        "Mean SHAP": mean_shap,
        "Lower Error": np.maximum(0, lower_bound),  
        "Upper Error": np.maximum(0, upper_bound),  
    }
)

df_mean_shap = df_mean_shap.sort_values(by="Mean SHAP", ascending=True)

fig, ax = plt.subplots(figsize=(12, 10), dpi=160)

features = df_mean_shap["Feature"]
mean_shap_values = df_mean_shap["Mean SHAP"]
y_positions = np.arange(len(features)) * 2  

ax.barh(
    y_positions,
    mean_shap_values,
    xerr=np.array([df_mean_shap["Lower Error"], df_mean_shap["Upper Error"]]),
    color="skyblue",
    capsize=3,
    edgecolor="black",
    alpha=0.8,
)

ax.set_yticks(y_positions)
ax.set_yticklabels(features, fontsize=12)

ax.set_xlabel("Mean SHAP Value (average impact on the outcome)", fontsize=14)
ax.set_ylabel("Features", fontsize=14)
ax.set_title("Global Feature Importance", fontsize=16)
ax.tick_params(axis="x", labelsize=12)
ax.grid(axis="x", linestyle="--", alpha=0.7)

plt.tight_layout()
output_path = r'D:\Figures\global_feature_importance_spaced.pdf'
plt.savefig(output_path, format="pdf", dpi=800, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

l_95 = np.percentile(shap_bootstrap_array, axis=0, q=2.5) 
u_95 = np.percentile(shap_bootstrap_array, axis=0, q=97.5)  

def plot(ax, term=0):
    
    order = np.argsort(X_test.values[:, term])

    ax.fill_between(
        X_test.values[:, term][order],
        l_95[:, term][order],
        u_95[:, term][order],
        color='lightblue',
        alpha=0.6
    )
    
    ax.scatter(
        X_test.values[:, term][order],
        shap_bootstrap_array.mean(axis=0)[:, term][order],
        s=10,
        color='black'
    )
    
    ax.axhline(0, color='red', linestyle='--', linewidth=1)
    
    ax.set_xlabel(X_test.columns[term], fontsize=13)
    ax.set_ylabel("SHAP value", fontsize=13)
    plt.tight_layout()

fig, ax = plt.subplots(3, 3, figsize=(12, 8), dpi=300)  
ax = ax.ravel() 


selected_features = [2, 7, 9, 8, 3, 5, 10, 4, 6]  


index = 0
for feature_idx in selected_features:
    plot(ax=ax[index], term=feature_idx)
    index += 1

plt.tight_layout()
plt.savefig(r'D:\Figures\global_feature_importance_with_intervals_test.pdf', format="pdf", dpi=800)
plt.show()
